In [ ]:
import os
import textwrap
from datetime import datetime

import chardet
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from IPython.display import display
from matplotlib.colors import Normalize

pd.set_option('display.max_colwidth', None)

# --- Seaborn Theme ---
sns.set_theme(style="whitegrid", palette="pastel", font_scale=1.2)
%config InlineBackend.figure_format = 'svg'

### Utility methods

In [ ]:
def detect_encoding(path):
        """Detect file encoding using chardet"""
        with open(path, 'rb') as f:
            raw = f.read(10000)
        result = chardet.detect(raw)
        enc = result.get("encoding")
        if enc:
            enc = enc.strip()
            print("🔍 Detected encoding: %s (confidence: %.2f)", enc, result.get("confidence", 0.0))
            return enc
        print("⚠️ Encoding detection failed, defaulting to latin-1.")
        return "latin-1"

def load_data(sub_path, eval_path, input_path):
    """Load CSV data into pandas DataFrames."""
    subs = pd.read_csv(sub_path)
    evals = pd.read_csv(eval_path)
    enc = detect_encoding(input_path)
    inputs = clean_inputs(pd.read_csv(input_path, encoding=enc))
    return subs, evals, inputs

def clean_inputs(inputs):
    # Remove any blank or non-numeric rows (like "Open-Ended Response")
    inputs = inputs[inputs["Respondent ID"].notna()].copy()

    # Normalize column names
    inputs.columns = inputs.columns.str.strip()

    # Rename and keep relevant columns
    EMAIL_FIELD = "Please share your email address so we can contact you if you win"

    inputs_clean = inputs.rename(columns={
        "Respondent ID": "respondent_id",
        EMAIL_FIELD: "email_address",
        "First Name": "first_name",
        "Last Name": "last_name"
    })[["respondent_id", "first_name", "last_name", "email_address"]]

    # --- Clean and convert respondent_id to int64 safely ---
    # remove spaces, non-digits, and trailing '.0'
    inputs_clean["respondent_id"] = (
        inputs_clean["respondent_id"]
        .astype(str)
        .str.strip()
        .str.replace(r"[^\d.]", "", regex=True)
        .str.replace(r"\.0$", "", regex=True)
    )

    # convert to numeric, coercing bad values to NaN, then drop them
    inputs_clean["respondent_id"] = pd.to_numeric(inputs_clean["respondent_id"], errors="coerce")
    inputs_clean = inputs_clean.dropna(subset=["respondent_id"])
    inputs_clean["respondent_id"] = inputs_clean["respondent_id"].astype("int64")

    return inputs_clean

In [ ]:
def get_comp_table(evals, inputs_clean):
    evals = evals.copy()
    evals["sub_id"] = evals["sub_id"]
    print("Evals sub_id dtype:", evals["sub_id"].dtype)
    print("Inputs respondent_id dtype:", inputs_clean["respondent_id"].dtype)
    print("Sample mismatched IDs:", set(evals["sub_id"]) - set(inputs_clean["respondent_id"]))


    comparison_table = (evals
        .merge(inputs_clean, left_on="sub_id", right_on="respondent_id", how="left")
        .drop(columns=["respondent_id"]))

    ids = ["eval_id", "sub_id"]
    scores = ["knowledge", "enthusiasm", "humor", "average"]

    comparison_table = comparison_table[["first_name", "last_name", "email_address"] + ids + scores]
    return comparison_table



def display_raw_data(df, title):
    """Display a nicely formatted dataframe with wrapped text."""
    styles = [
        # Caption styling
        dict(selector='caption', props=[('font-size', '16px'), ('font-weight', 'bold')]),
        # Make text truncate instead of wrap
        dict(selector='td', props=[('white-space', 'nowrap'), ('overflow', 'hidden'), ('text-overflow', 'ellipsis'), ('max-width', '400px')]),
        dict(selector='th', props=[('white-space', 'nowrap'), ('overflow', 'hidden'), ('text-overflow', 'ellipsis'), ('max-width', '200px')]),
    ]

    display(df.head(10).style
            .set_caption(title)
            .set_table_styles(styles))

def add_average_column(df):
    """Add average score column right after 'humor'."""
    # Compute the average
    df["average"] = df[["knowledge", "enthusiasm", "humor"]].mean(axis=1)
    # Find the index position of 'humor'
    humor_index = df.columns.get_loc("humor")
    # Reorder columns so 'average' comes right after 'humor'
    cols = df.columns.tolist()
    # Move 'average' to position after 'humor'
    cols.insert(humor_index + 1, cols.pop(cols.index("average")))
    return df[cols]

In [ ]:
def get_topn(df, column, n):
    """Return top 10 entries by a given score."""
    return df.sort_values(by=column, ascending=False).head(n)

def get_missing_eval(subs, evals):
    """Return submissions missing any of the evaluation scores."""
    missing = evals[evals[["knowledge", "enthusiasm", "humor"]].isnull().any(axis=1)]
    missing_ids = subs[~subs["sub_id"].isin(evals["sub_id"])]
    return pd.concat([missing, missing_ids]).drop_duplicates(subset=["sub_id"])

def get_duplicate_eval(evals):
    """Return duplicate evaluations based on sub_id."""
    dupes = evals[evals.duplicated(subset=["sub_id"], keep=False)]
    return dupes.sort_values(by="sub_id")

def wrap_text(text, width=60):
    if isinstance(text, str):
        return "\n".join(textwrap.wrap(text, width=width))
    return text

def export_to_csv(df, name, out_dir="exports"):
    """
    Export a given DataFrame to a timestamped CSV file.
    """
    os.makedirs(out_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = os.path.join(out_dir, f"{name}_{timestamp}.csv")
    df.to_csv(file_path, index=False)
    print(f"💾 Exported '{name}' → {file_path}")
    return file_path

### Main flow

In [ ]:
# --- Load Data ---
subs, evals, inputs = load_data("output/sub.csv", "output/eval.csv", "output/inputs.csv")

In [ ]:
ids = ["eval_id", "sub_id"]
scores = ["knowledge", "enthusiasm", "humor", "average"]
bd = ["brief_description"]
perf = ["total_tokens", "prompt_tokens", "completion_tokens", "successful_requests",
        "total_cost", "time_taken_in_seconds", "total_processing_time_in_seconds", "evaluated_at"]

In [ ]:
# --- Add Average ---
evals = add_average_column(evals)
evals["brief_description"] = evals["brief_description"].apply(wrap_text)

In [ ]:
# --- Display Raw Tables ---
display_raw_data(subs.head(), "Submissions (sub.csv)")

In [ ]:
display_raw_data(evals.head(), "Evaluations (eval.csv)")

In [ ]:
# --- Display Raw Tables ---
display_raw_data(inputs.head(), "Inputs (inputs.csv)")

### Missing and Dupes

In [ ]:
# --- Missing Evaluations ---
missing_eval = get_missing_eval(subs, evals)
display_raw_data(missing_eval, title="Missing Evaluations")

In [ ]:
# --- Duplicate Evaluations ---
duplicate_eval = get_duplicate_eval(evals)
display_raw_data(duplicate_eval, title="Duplicate Evaluations")

### Comparison

In [ ]:
# --- Comparison Tables ---
comparison_table = get_comp_table(evals, inputs)
display_raw_data(comparison_table, "Comparison Table: All Submissions")

In [ ]:
# --- Top 10 Tables ---
subset = evals[ids + scores].copy()
for col in scores:
    top = get_topn(subset, col, n=5)
    display_raw_data(top, f"Top players by {col.title()}")

### Visualization methods

In [ ]:
def plot_scores_distribution(df, same_y_scale=True, fig_width=18, fig_height=4, bins=10,
                             axis_label_fontsize=11, tick_fontsize=9, cmap="viridis"):
    """
    Plot score distributions as 4 histograms in a 1x4 grid.
    """
    score_cols = ["knowledge", "enthusiasm", "humor", "average"]

    # Compute shared y-axis limit if needed
    y_max = None
    if same_y_scale:
        y_max = max(df[col].value_counts(bins=bins).max() for col in score_cols)

    fig, axes = plt.subplots(1, 4, figsize=(fig_width, fig_height))

    # Prepare color map
    cmap_obj = plt.get_cmap(cmap)
    norm = Normalize(vmin=0, vmax=bins - 1)

    for ax, col in zip(axes, score_cols, strict=False):
        # Compute histogram manually to access bins and heights
        counts, bin_edges = np.histogram(df[col].dropna(), bins=bins)
        bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])

        # Assign a color to each bin
        colors = [cmap_obj(norm(i)) for i in range(bins)]

        # Draw colored bars
        ax.bar(bin_centers, counts, width=np.diff(bin_edges), align='center',
               edgecolor='black', color=colors)

        # Titles & labels
        ax.set_title(col.title(), fontsize=12, weight='bold')
        ax.set_xlabel("Score", fontsize=axis_label_fontsize)
        ax.set_ylabel("Frequency", fontsize=axis_label_fontsize)
        ax.tick_params(axis='both', which='major', labelsize=tick_fontsize)
        ax.grid(alpha=0.3)
        if same_y_scale and y_max is not None:
            ax.set_ylim(0, y_max)

    fig.suptitle("Score Distributions", fontsize=16, weight='bold', y=1.05)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Plots ---
plot_scores_distribution(df=evals, fig_width=15, axis_label_fontsize=12, cmap="Pastel1")

In [ ]:
def plot_box_distribution(df, fig_width=12, fig_height=6, axis_label_fontsize=11, tick_fontsize=9, title_fontsize=16, ylim=None):
    """
    Plot box plot for score distribution across criteria.
    """
    melted = df.melt(
        id_vars=["sub_id"],
        value_vars=["knowledge", "enthusiasm", "humor", "average"],
        var_name="criteria",
        value_name="score"
    )

    plt.figure(figsize=(fig_width, fig_height))
    sns.boxplot(data=melted,
                x="criteria",
                y="score",
                hue="criteria",
                palette="pastel",
                legend=False)

    plt.title("Scores per Evaluation Criteria", fontsize=title_fontsize, weight="bold")
    plt.xlabel("Criteria", fontsize=axis_label_fontsize)
    plt.ylabel("Score", fontsize=axis_label_fontsize)
    plt.xticks(fontsize=tick_fontsize)
    plt.yticks(fontsize=tick_fontsize)
    plt.grid(alpha=0.3)

    if ylim is not None:
        plt.ylim(ylim)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_box_distribution(df=evals, axis_label_fontsize=12, ylim=(0,100))

In [ ]:
def plot_radar_chart_plotly(df, sub_ids, criteria=None, fig_size=(700, 600), title_fontsize=18,
    tick_fontsize=11, label_fontsize=13, ylim=None, palette="Set2", alpha_fill=0.25, max_legends=20):
    """
    Create an interactive radar (spider) chart comparing multiple submissions using Plotly.
    """
    if criteria is None:
        criteria = ["knowledge", "enthusiasm", "humor", "average"]

    # Filter relevant submissions and set index
    data = df[df["sub_id"].isin(sub_ids)][["sub_id"] + criteria].set_index("sub_id")

    # Close the radar loop
    categories = criteria + [criteria[0]]

    # Use Plotly palette safely
    colors = px.colors.qualitative.__dict__.get(palette, px.colors.qualitative.Set2)
    colors = colors * (len(sub_ids) // len(colors) + 1)

    # Create figure
    fig = go.Figure()

    for i, (sub_id, row) in enumerate(data.iterrows()):
        values = row.tolist() + [row.tolist()[0]]
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=categories,
            fill='toself',
            fillcolor=colors[i].replace("rgb", "rgba").replace(")", f", {alpha_fill})"),
            line=dict(color=colors[i], width=2),
            name=f"Sub {sub_id}",
            hovertemplate="<b>%{{theta}}</b>: %{{r}}<extra>Sub {}</extra>".format(sub_id)
        ))

    # Axis limits
    r_min = ylim[0] if ylim else float(data.min().min())
    r_max = ylim[1] if ylim else float(data.max().max())

    # Layout styling
    fig.update_layout(
        polar=dict(
            bgcolor='white',
            radialaxis=dict(
                visible=True,
                range=[r_min, r_max],
                showline=False,
                gridcolor='lightgray',
                gridwidth=1,
                tickfont=dict(size=tick_fontsize)
            ),
            angularaxis=dict(
                tickfont=dict(size=label_fontsize, color="black")
            ),
        ),
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.15,
            xanchor="center",
            x=0.5,
            font=dict(size=tick_fontsize),
            itemsizing='trace'
        ),
        title=dict(
            text="Radar Chart Comparison",
            x=0.5,
            y=0.95,
            xanchor='center',
            yanchor='top',
            font=dict(size=title_fontsize, color='black')
        ),
        width=fig_size[0],
        height=fig_size[1],
        template="plotly_white",
        margin=dict(l=40, r=40, t=80, b=80)
    )

    # Handle too many legends gracefully
    if len(sub_ids) > max_legends:
        for i, trace in enumerate(fig.data):
            trace.showlegend = i < max_legends
        fig.add_annotation(
            text=f"... (+{len(sub_ids) - max_legends} more)",
            x=0.5, y=-0.25,
            showarrow=False,
            xref="paper", yref="paper",
            font=dict(size=tick_fontsize, color="gray")
        )

    fig.show()

In [ ]:
top_overall = get_topn(subset, "average", n=5)
sub_ids = np.array(top_overall["sub_id"])[:3]

plot_radar_chart_plotly(evals, sub_ids=sub_ids, ylim=(0, 100),
    fig_size=(800, 700), palette="Set2", alpha_fill=0.3, max_legends=10)

### End of report